# Day 27 · Learning rate: too small, too big, just right

**First-principles question:** Why does training sometimes explode or crawl?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 40 months of (fake) hosting data. The same every time you run it.
rng = np.random.default_rng(42)
x = rng.uniform(0.05, 2.0, 40).round(2)                     # requests per month, in lakhs (1 lakh = 100,000)
y = (2.5 + 1.8 * x + rng.normal(0, 0.25, 40)).round(2)      # hosting cost per month, in ₹ thousands

def train(x, y, learning_rate, steps=300):
    """Gradient descent with the exact gradient (day 6 derives it). Returns the loss after every step."""
    w, b, losses = 0.0, 0.0, []
    for _ in range(steps):
        error = w * x + b - y
        w -= learning_rate * float(np.mean(2 * error * x))
        b -= learning_rate * float(np.mean(2 * error))
        losses.append(float(np.mean((w * x + b - y) ** 2)))
        if not np.isfinite(losses[-1]) or losses[-1] > 1e12:
            break
    return losses

## 1 · Four learning rates

In [ ]:
# TODO: pick 4 learning rates: one that crawls, two that work, one that explodes (try 0.001 … 1.0)
RATES = [0.001, None, None, None]

for lr in RATES:
    plt.plot(train(x, y, lr), label=f"lr = {lr}")
plt.yscale("log"); plt.ylim(1e-2, 1e3); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

## 2 · The same data in other units
Requests instead of lakhs (× 100,000) and ₹ instead of ₹ thousands (× 1,000). Same business, bigger numbers.

In [ ]:
x_raw, y_raw = x * 100_000, y * 1000
# TODO: train on (x_raw, y_raw) with one of your good learning rates. What happens to the loss?
raw_losses = None
print("first losses:", raw_losses[:5])

## 3 · Steps needed

In [ ]:
best = min(train(x, y, 0.1, steps=3000))
def steps_needed(lr, tolerance=1.05):
    losses = train(x, y, lr, steps=3000)
    for i, loss in enumerate(losses):
        if loss <= best * tolerance:
            return i + 1
    return None      # never got there

for lr in RATES:
    print(f"lr = {lr}: {steps_needed(lr)} steps to get within 5% of the best loss")

**What I would check first if a customer's training 'blows up':** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(RATES) == 4 and None not in RATES, "Step 1: four learning rates"
finals = [train(x, y, lr)[-1] for lr in RATES]
assert any((not np.isfinite(f)) or f > 1e3 for f in finals), "Step 1: one learning rate should explode"
assert any(np.isfinite(f) and f < 0.1 for f in finals), "Step 1: at least one should work"
assert raw_losses is not None and len(raw_losses) > 0, "Step 2: train on the raw units"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")